# 01 — REST Setup: Projects & Collections (SDK-first)

Authenticate with the SDK, create a project + collection, verify list/get, clean up.
Runnable top-to-bottom. Skips cleanly without creds or without the `nostradamus` org.

In [ ]:
# Cell 1: install (pin SHA deliberately, never track main)
%pip install -q "nostradamus-ioto-sdk @ git+https://github.com/robotics-4-all/nostradamus-ioto-sdk.git@4f8e7fc7c28c859eefed9f7382d1b148a5c6eeb8" python-dotenv

In [ ]:
import os, random, time
from dotenv import load_dotenv
load_dotenv()

BASE_URL = os.getenv("CENOTOO_BASE_URL", "http://localhost:8000")
USERNAME = os.getenv("CENOTOO_USERNAME", "")
PASSWORD = os.getenv("CENOTOO_PASSWORD", "")

if not (BASE_URL and USERNAME and PASSWORD):
    print("SKIP: set CENOTOO_BASE_URL, CENOTOO_USERNAME, CENOTOO_PASSWORD")
    CREDS_OK = False
else:
    CREDS_OK = True
    print(f"Target: {BASE_URL}")

def unique_name(prefix="nb"):
    stamp = time.strftime("%Y%m%d%H%M%S")
    return f"{prefix}_{stamp}_{random.randint(1000,9999)}"

In [ ]:
assert CREDS_OK, "stopping: no creds"
from nostradamus_ioto_sdk import NostradamusClient
from nostradamus_ioto_sdk.exceptions import ResourceNotFoundError

client = NostradamusClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
print("authenticated")

# Org guard: servers seeded without 'nostradamus' answer 404 -> skip, don't fail
try:
    org = client.organizations.get()
    print("org:", org)
except ResourceNotFoundError:
    print("SKIP: default organization 'nostradamus' not present on this server")
    client.close()
    raise SystemExit(0)

In [ ]:
project = client.projects.create(name=unique_name("nb_proj"), description="notebook 01 project", tags=["notebook", "01"])
print("project:", project.project_id, project.name)

In [ ]:
collection = client.collections.create(
    project_id=project.project_id,
    name=unique_name("nb_coll"),
    description="notebook 01 collection",
    tags=["notebook", "01"],
    collection_schema={"key": "NB_001", "timestamp": "2026-01-01T00:00:00Z", "temperature": 20.0, "humidity": 50.0},
)
print("collection:", collection.collection_id, collection.name)

In [ ]:
got_p = client.projects.aget if False else client.projects.get(project.project_id)
assert str(got_p.project_id) == str(project.project_id)
got_c = client.collections.get(project.project_id, collection.collection_id)
assert str(got_c.collection_id) == str(collection.collection_id)
print("list/get verified")

# Negative: bogus id must 404
try:
    client.projects.get("00000000-0000-0000-0000-000000000000")
    raise AssertionError("expected ResourceNotFoundError")
except ResourceNotFoundError:
    print("negative OK: bogus project -> 404")

In [ ]:
import contextlib
with contextlib.suppress(Exception):
    client.collections.delete(project.project_id, collection.collection_id)
    print("collection deleted")
with contextlib.suppress(Exception):
    client.projects.delete(project.project_id)
    print("project deleted")
client.close()
print("cleanup done — no residue")